# Machine Learning Pipeline: Resnet
**Predicting Bloodstream Infections**

This notebook is part of the official codebase for the paper: *"[Machine learning pipeline for predicting positive blood cultures using Sysmex hematology data in the emergency department.]"*. 
It contains the complete end-to-end machine learning pipeline for training, tuning, and evaluating the **Resnet** model.

### Overview of this notebook:
1. **Data Loading & Preprocessing:** Formatting the numerical dataset for model ingestion.
2. **Feature Selection:** Identifying predictive clinical variables (options: RFE, Boruta, Lasso, Tree Importance, or None).
3. **Hyperparameter Tuning:** Bayesian optimization using Optuna.
4. **Model Training & Calibration:** Training with cross-validation and applying Sigmoid probability calibration.
5. **Evaluation:** Extensive metric calculation, bootstrapping, SHAP value analysis, and threshold optimization.

**Repository & Citation:** 
For the full codebase, installation instructions, and citation details, please visit the [GitHub Repository](https://github.com/UMCG-SEH/blood-culture-outcome-classification).

*Note: Run all cells sequentially. Interactive prompts will appear at the bottom to configure the pipeline settings (e.g., device selection, validation split, SMOTE).*

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import json
from copy import deepcopy
from typing import List, Dict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

import joblib

from sklearn.model_selection import (
    StratifiedShuffleSplit, 
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.calibration import cross_val_predict
from sklearn.exceptions import FitFailedWarning
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, 
    accuracy_score, 
    confusion_matrix, 
    recall_score, 
    roc_curve, 
    auc, 
    make_scorer, 
    precision_recall_curve, 
    average_precision_score,
    precision_score, 
    balanced_accuracy_score, 
    f1_score, 
    fbeta_score, 
    matthews_corrcoef,
    log_loss
)

from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN, SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.base import BaseEstimator, ClassifierMixin
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
import optuna

from sklearn.calibration import CalibratedClassifierCV

import torch
print(torch.__version__)
print(torch.cuda.is_available()) 

from joblib import Parallel, delayed


In [162]:
class ResBlock(nn.Module):
    """
    Residual fully-connected block for tabular data.

    This block consists of two linear layers with batch normalization,
    ReLU activations, dropout, and a residual (shortcut) connection.
    """
    def __init__(self, in_dim, out_dim, dropout=0.1):
        """
        Initialize a residual block.

        Args:
            in_dim: Input feature dimension.
            out_dim: Output feature dimension.
            dropout: Dropout probability.
        """
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim

        self.fc1 = nn.Linear(in_dim, out_dim)
        self.bn1 = nn.BatchNorm1d(out_dim)
        self.fc2 = nn.Linear(out_dim, out_dim)
        self.bn2 = nn.BatchNorm1d(out_dim)

        self.dropout = nn.Dropout(dropout)
        if in_dim != out_dim:
            self.shortcut = nn.Linear(in_dim, out_dim)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        """
        Forward pass through the residual block.

        Args:
            x: Input tensor of shape (batch_size, in_dim).

        Returns:
            Tensor of shape (batch_size, out_dim).
        """
        residual = self.shortcut(x)

        out = self.fc1(x)
        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout(out)

        out = self.fc2(out)
        out = self.bn2(out)
        out = F.relu(out)

        out = out + residual
        return out

class TabularResNet(nn.Module):
    """
    Residual neural network for tabular data.

    The network consists of an input layer, multiple residual blocks,
    and a single-output layer producing logits for binary classification.
    """

    def __init__(self, input_dim, hidden_dim=64, n_blocks=3, dropout=0.1):
        """
        Initialize the tabular ResNet architecture.

        Args:
            input_dim: Number of input features.
            hidden_dim: Hidden layer dimension.
            n_blocks: Number of residual blocks.
            dropout: Dropout probability.
        """
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([
            ResBlock(hidden_dim, hidden_dim, dropout) for _ in range(n_blocks)
        ])
        self.output_layer = nn.Linear(hidden_dim, 1)  

    def forward(self, x):
        """
        Forward pass through the network.

        Args:
            x: Input tensor of shape (batch_size, input_dim).

        Returns:
            Logits tensor of shape (batch_size, 1).
        """
        x = self.input_layer(x)
        for block in self.blocks:
            x = block(x)
        x = self.output_layer(x)
        return x

class ResNetClassifier(BaseEstimator, ClassifierMixin):
    """
    Scikit-learn compatible classifier wrapper for a tabular ResNet model.

    This class provides fit, predict, and predict_proba methods and supports
    early stopping, GPU acceleration, and model serialization.
    """

    def __init__(self, input_dim, hidden_dim=64, n_blocks=3, dropout=0.1,
                 lr=1e-3, batch_size=64, epochs=50, device='CPU', verbose=True,
                 random_state=42):
        """
        Initialize the ResNet classifier.

        Args:
            input_dim: Number of input features.
            hidden_dim: Hidden layer dimension.
            n_blocks: Number of residual blocks.
            dropout: Dropout probability.
            lr: Learning rate.
            batch_size: Training batch size.
            epochs: Maximum number of training epochs.
            device: 'CPU' or 'GPU'.
            verbose: Whether to print training progress.
            random_state: Random seed for reproducibility.
        """
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.n_blocks = n_blocks
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.device = device
        self.verbose = verbose
        self.random_state = random_state
        self.model = None

    def fit(self, X, y, X_val=None, y_val=None, patience=10):
        """
        Train the ResNet classifier.

        Optionally applies early stopping based on validation loss.

        Args:
            X: Training feature matrix.
            y: Training labels.
            X_val (optional): Validation feature matrix.
            y_val (optional): Validation labels.
            patience (int): Number of epochs without improvement before
                early stopping.

        Returns:
            self: Trained classifier instance.
        """
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        device = torch.device('cuda' if self.device=='GPU' and torch.cuda.is_available() else 'cpu')

        self.model = TabularResNet(self.input_dim, self.hidden_dim, self.n_blocks, self.dropout).to(device)

        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=1e-5)
        criterion = nn.BCEWithLogitsLoss()

        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        y_tensor = torch.tensor(y.reshape(-1,1), dtype=torch.float32).to(device)
        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        loader = torch.utils.data.DataLoader(dataset,batch_size=self.batch_size,shuffle=True,drop_last=True)


        if X_val is not None and y_val is not None:
            X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
            y_val_t = torch.tensor(y_val.reshape(-1,1), dtype=torch.float32).to(device)

        best_loss = np.inf
        best_state = None
        no_improve = 0

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            epoch_loss = 0.0
            for xb, yb in loader:
                optimizer.zero_grad()
                out = self.model(xb)
                loss = criterion(out, yb)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * xb.size(0)
            epoch_loss /= len(loader.dataset)

            if X_val is not None and y_val is not None:
                self.model.eval()
                with torch.no_grad():
                    val_logits = self.model(X_val_t)
                    val_loss = criterion(val_logits, y_val_t).item()

                if val_loss < best_loss - 1e-4:
                    best_loss = val_loss
                    best_state = deepcopy(self.model.state_dict())
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        if self.verbose:
                            print(f"Early stopping at epoch {epoch} (val loss {val_loss:.4f})")
                        break
            if self.verbose:
                print(f"Epoch {epoch}/{self.epochs} — Train loss: {epoch_loss:.4f}")

        if best_state is not None:
            self.model.load_state_dict(best_state)

        return self


    def predict_proba(self, X):
        """
        Predict class probabilities for input samples.

        Args:
            X: Feature matrix.

        Returns:
            np.ndarray: Array of shape (n_samples, 2) containing
            class probabilities.
        """
        device = torch.device('cuda' if self.device=='GPU' and torch.cuda.is_available() else 'cpu')
        self.model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = self.model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy()
        return np.hstack([1 - probs, probs])

    def predict(self, X):
        """
        Predict binary class labels for input samples.

        Args:
            X: Feature matrix.

        Returns:
            np.ndarray: Predicted class labels (0 or 1).
        """
        probs = self.predict_proba(X)[:,1]
        return (probs >= 0.5).astype(int)

    def __getstate__(self):
        """
        Customize object serialization to include PyTorch model weights.

        Returns:
            dict: Serializable state dictionary.
        """
        state = self.__dict__.copy()
        model = state.pop('model', None)
        if model is not None:
            if isinstance(model, nn.DataParallel):
                model_to_save = model.module
            else:
                model_to_save = model
            state['_torch_state_dict'] = model_to_save.state_dict()
        else:
            state['_torch_state_dict'] = None
        return state

    def __setstate__(self, state):
        """
        Restore object state including PyTorch model weights.

        Args:
            state: Serialized state dictionary.
        """
        _sd = state.pop('_torch_state_dict', None)
        self.__dict__.update(state)
        if _sd is not None:
            model = TabularResNet(self.input_dim, self.hidden_dim, self.n_blocks, self.dropout)
            model.load_state_dict(_sd)
            self.model = model
        else:
            self.model = None


In [163]:
def make_output_folder(base: str, name: str) -> str:
    """
    Create a folder at the specified location if it doesn't exist.

    Args:
        base (str): Base directory where the folder should be created.
        name (str): Name of the folder to create.

    Returns:
        str: Full path to the created (or existing) folder.
    """
    path = os.path.join(base, name)
    os.makedirs(path, exist_ok=True)
    return path

In [164]:
def save_model(path: str, pipeline):
    """
    Save a model object to the specified folder using joblib.

    Args:
        path (str): Folder path where the model will be saved.
        model: The model object to save.

    Returns:
        None
    """
    joblib.dump(pipeline, os.path.join(path, 'pipeline.joblib'))


In [165]:
def plot_roc(y_true, y_proba, out_path: str):
    """
    Plot the ROC curve for a set of true labels and predicted probabilities, 
    and save it as 'roc.png' in the specified folder.

    Args:
        y_true: true binary labels.
        y_proba: predicted probabilities for the positive class.
        out_path (str): Folder path where the ROC plot will be saved.

    Returns:
        None
    """
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC curve (AUC = {roc_auc:.4f})')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, 'roc.png'))
    plt.close()

In [166]:
def plot_pr_curve(y_true, y_proba, out_path: str):
    """
    Plot the Precision-Recall curve for a set of true labels and predicted probabilities, 
    and save it as 'pr_curve.png' in the specified folder.

    Args:
        y_true: true binary labels.
        y_proba: predicted probabilities for the positive class.
        out_path (str): Folder path where the PR curve plot will be saved.

    Returns:
        None
    """
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    ap_score = average_precision_score(y_true, y_proba)

    plt.figure()
    plt.plot(recall, precision, label=f'AP = {ap_score:.4f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall curve')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, 'pr_curve.png'))
    plt.close()

In [167]:
def plot_confusion(y_true, y_pred, out_path: str):
    """
    Plot a confusion matrix for true vs predicted labels and save it as 'confusion.png'.

    Args:
        y_true: true binary labels.
        y_pred: predicted binary labels.
        out_path (str): Folder path where the confusion matrix plot will be saved.

    Returns:
        None
    """ 
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title('Confusion matrix')
    plt.colorbar()
    
    classes = ['neg','pos']
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)
    
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'), horizontalalignment="center")
    
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, 'confusion.png'))
    plt.close()

In [168]:
def evaluate_model(clf, X_test, y_test) -> Dict[str, float]:
    """
    Evaluate a classification model on test data and return key metrics.

    Args:
        clf: Trained classifier with predict and predict_proba methods.
        X_test: Test features.
        y_test: True labels for the test set.

    Returns:
        Dict[str, float]: Dictionary containing AUC, accuracy, sensitivity, specificity,
                          confusion matrix counts (TP, TN, FP, FN), and combined metric.
    """
    y_prob = clf.predict_proba(X_test)[:, 1]
    y_pred = clf.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    sens = recall_score(y_test, y_pred, pos_label=1)
    spec = recall_score(y_test, y_pred, pos_label=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    combined = (sens + spec) / 2
    
    return {
        'auc': float(auc_score),
        'accuracy': float(acc),
        'sensitivity': float(sens),
        'specificity': float(spec),
        'tp': int(tp),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'combined': float(combined)
    }


In [169]:
def select_features(method, X_train, y_train, X_test, feature_columns, out_path=None, top_k=20, scoring_metric='roc_auc'):
    """
    Select features from the training and test sets based on the specified method.

    Currently, only the 'NONE' method is supported, which uses all available
    features without applying any selection. The selected feature names can
    optionally be saved to disk.

    Args:
        method: Feature selection method. Use 'NONE' to disable
            feature selection.
        X_train: Training feature matrix.
        y_train: Training target labels (currently unused).
        X_test: Test feature matrix.
        feature_columns: List of feature names corresponding to the columns
            in X_train and X_test.
        out_path: Folder path to save the selected feature names.
        top_k: Number of top features to select (currently unused).
        scoring_metric: Scoring metric used for feature selection
            (currently unused).

    Returns:
        X_train_sel: Selected training feature matrix.
        X_test_sel: Selected test feature matrix.
        selected_feat_names: List of selected feature names.
    """
    X_train_sel = np.array(X_train)
    X_test_sel = np.array(X_test)
    selected_feat_names = feature_columns

    method = method.upper()
    if method != 'NONE':
        print(f"Warning: feature selection method '{method}' is not supported. Use 'NONE'.")
    
    if out_path:
        feat_df = pd.DataFrame({'selected_features': selected_feat_names})
        os.makedirs(out_path, exist_ok=True)
        feat_df.to_csv(os.path.join(out_path, 'selected_features_NONE.csv'), index=False)
        print(f"All features used, saved to: {os.path.join(out_path, 'selected_features_NONE.csv')}")

    if X_train_sel.ndim == 1:
        X_train_sel = X_train_sel.reshape(-1, 1)
        X_test_sel = X_test_sel.reshape(-1, 1)

    return X_train_sel, X_test_sel, selected_feat_names


In [170]:
def build_pipeline(smote_type=None, categorical_features=None, device='CPU', input_dim=None):
    """
    Build a machine learning pipeline with optional SMOTE resampling and a resnet classifier.

    Args:
        smote_type (optional): Type of SMOTE resampling to apply. Options: 
            'SMOTE', 'BorderlineSMOTE', 'ADASYN', 'SMOTENC'. Defaults to None.
        scale_pos_weight (optional): Class weight for the positive class in xgboost. Defaults to 1.
        categorical_features: Indices of categorical features for SMOTENC. Required if smote_type='SMOTENC'.
        device (optional): Device for resnet training. Options: 'CPU', 'GPU'. Defaults to 'CPU'.

    Returns:
        ImbPipeline: Pipeline object including optional SMOTE, and resnet classifier.
    """
    steps = [('scaler', StandardScaler())]

    if smote_type is not None:
        st = str(smote_type).strip().upper()
        if st != 'NONE':
            if st == 'SMOTE':
                sampler = SMOTE(random_state=42)
            elif st in ('BORDERLINE', 'BORDERLINESMOTE', 'BORDERLINE_SMOTE'):
                sampler = BorderlineSMOTE(random_state=42)
            elif st == 'ADASYN':
                sampler = ADASYN(random_state=42)
            elif st in ('SMOTENC', 'SMOTE-NC', 'SMOTE_NC'):
                if categorical_features is None:
                    raise ValueError("For SMOTENC categorical_features indices must be specified.")
                sampler = SMOTENC(categorical_features=categorical_features, random_state=42)
            else:
                raise ValueError(f"Unknown SMOTE-type: {smote_type}")
            steps.append(('smote', sampler))

    resnet = ResNetClassifier(
        input_dim=input_dim,
        hidden_dim=64,
        n_blocks=3,
        dropout=0.1,
        lr=1e-3,
        batch_size=64,
        epochs=50,
        device=device
    )
    steps.append(('resnet', resnet))

    return ImbPipeline(steps=steps)


In [171]:
def get_scorer_fixed(scoring_metric):
    """
    Return a scikit-learn compatible scorer for the specified evaluation metric.

    This function ensures correct handling of probability-based metrics by
    extracting the positive class probability when required.

    Args:
        scoring_metric: Name of the scoring metric (e.g. 'roc_auc',
            'average_precision', 'accuracy', 'f1').

    Returns:
        scorer: Scikit-learn scorer object created with make_scorer.

    Raises:
        ValueError: If the specified scoring metric is not supported.
    """
    scoring_metric = scoring_metric.lower()
    if scoring_metric == 'roc_auc':
        return make_scorer(
            lambda y_true, y_pred, **kwargs: roc_auc_score(y_true, 
                                                           y_pred[:,1] if y_pred.ndim > 1 else y_pred),
            needs_proba=True
        )
    elif scoring_metric in ['average_precision', 'ap']:
        return make_scorer(
            lambda y_true, y_pred, **kwargs: average_precision_score(y_true, 
                                                                     y_pred[:,1] if y_pred.ndim > 1 else y_pred),
            needs_proba=True
        )
    elif scoring_metric == 'logloss':
        return make_scorer(
            lambda y_true, y_pred, **kwargs: log_loss(y_true, 
                                                      y_pred[:,1] if y_pred.ndim > 1 else y_pred),
            needs_proba=True
        )
    elif scoring_metric == 'accuracy':
        return make_scorer(accuracy_score)
    elif scoring_metric == 'f1':
        return make_scorer(f1_score)
    elif scoring_metric == 'balanced_accuracy':
        return make_scorer(balanced_accuracy_score)
    elif scoring_metric == 'f2':
        return make_scorer(fbeta_score, beta=2)
    elif scoring_metric == 'mcc':
        return make_scorer(matthews_corrcoef)
    elif scoring_metric == 'recall':
        return make_scorer(recall_score)
    else:
        raise ValueError(f"Unsupported scoring metric: {scoring_metric}")


In [172]:
def compute_score(y_true, y_pred, scoring_metric):
    """
    Compute a classification performance score based on the selected metric.
    
    Args:
        y_true: true binary labels.
        y_pred: predicted labels or probabilities, depending on the metric.
        scoring_metric: Metric to compute. Options include:
            'roc_auc', 'average_precision'/'ap', 'logloss', 'accuracy', 'f1', 
            'balanced_accuracy', 'f2', 'mcc', 'recall'.
    
    Returns:
        float: Computed score for the given metric.
    
    Raises:
        ValueError: If an unsupported scoring metric is provided.
    """
    scoring_metric = scoring_metric.lower()
    
    if y_pred.ndim == 1 and np.any((y_pred > 0) & (y_pred < 1)):
        y_pred = (y_pred >= 0.5).astype(int)
    
    if scoring_metric == 'roc_auc':
        return roc_auc_score(y_true, y_pred)
    
    elif scoring_metric in ['average_precision', 'ap']:
        return average_precision_score(y_true, y_pred)
    
    elif scoring_metric == 'logloss':
        return log_loss(y_true, y_pred)
    
    elif scoring_metric == 'accuracy':
        return accuracy_score(y_true, y_pred)
    
    elif scoring_metric == 'f1':
        return f1_score(y_true, y_pred)
    
    elif scoring_metric == 'balanced_accuracy':
        return balanced_accuracy_score(y_true, y_pred)
    
    elif scoring_metric == 'f2':
        return fbeta_score(y_true, y_pred, beta=2)
    
    elif scoring_metric == 'mcc':
        return matthews_corrcoef(y_true, y_pred)
    
    elif scoring_metric == 'recall':
        return recall_score(y_true, y_pred)
    
    else:
        raise ValueError(f"Unsupported scoring_metric: {scoring_metric}")


In [ ]:
def optuna_objective(trial, X_train, y_train, X_val, y_val, smote_type, device, input_dim, scoring_metric='roc_auc', enable_pruning=False):
    """
    Optuna objective function for hyperparameter optimization of a ResNet-based
    classification pipeline.

    The function builds a pipeline, samples hyperparameters using the Optuna
    trial object, and evaluates model performance using cross-validation or
    a hold-out validation set.

    Args:
        trial: Optuna trial object used to sample hyperparameters.
        X_train: Training feature matrix.
        y_train: Training target labels.
        X_val (optional): Validation feature matrix.
        y_val (optional): Validation target labels.
        smote_type: SMOTE strategy used in the pipeline.
        device: Computation device ('CPU' or 'GPU').
        input_dim: Number of input features.
        scoring_metric: Metric used to evaluate performance
            (e.g. 'roc_auc', 'average_precision').
        enable_pruning: Whether to enable Optuna trial pruning.

    Returns:
        score: Evaluation score for the current trial.
    """
    params = {
        "resnet__epochs": trial.suggest_int("epochs", 50, 400),
        "resnet__lr": trial.suggest_float("lr", 1e-4, 5e-2, log=True),
        "resnet__batch_size": trial.suggest_int("batch_size", 256, 2048, log=True),
        "resnet__hidden_dim": trial.suggest_int("hidden_dim", 32, 512, log=True),
        "resnet__dropout": trial.suggest_float("dropout", 0.0, 0.4),
    }

    pipeline = build_pipeline(
        smote_type=smote_type,
        categorical_features=[],
        device=device,
        input_dim=input_dim
    )
    pipeline.set_params(**params)

    if X_val is None or y_val is None:
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scorer = get_scorer_fixed(scoring_metric)
        scores = cross_val_score(pipeline, X_train, y_train, cv=skf, scoring=scorer, n_jobs=1)
        score = float(np.mean(scores))
    else:
        pipeline.fit(X_train, y_train)
        scoring_metric_lower = scoring_metric.lower()
        if scoring_metric_lower in ['roc_auc', 'average_precision', 'ap', 'logloss']:
            y_pred = pipeline.predict_proba(X_val)[:,1]
        else:
            y_pred = pipeline.predict(X_val)
        score = float(compute_score(y_val, y_pred, scoring_metric))

    if enable_pruning:
        trial.report(score, step=0)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return score

In [174]:
def run_optuna_resnet(X_train, y_train, X_val=None, y_val=None, smote_type="SMOTE",
                      device="CPU", n_trials=200, scoring_metric='roc_auc', enable_pruning=False):
    """
    Run Optuna hyperparameter optimization for a ResNet-based
    tabular classification pipeline.

    The function creates and executes an Optuna study to maximize
    the specified scoring metric using a TPE sampler and optional
    trial pruning.

    Args:
        X_train: Training feature matrix.
        y_train: Training target labels.
        X_val (optional): Validation feature matrix.
        y_val (optional): Validation target labels.
        smote_type: SMOTE strategy used in the pipeline.
        device: Computation device ('CPU' or 'GPU').
        n_trials: Number of Optuna trials to run.
        scoring_metric: Metric to optimize (e.g. 'roc_auc').
        enable_pruning: Whether to enable Optuna pruning.

    Returns:
        study: Optuna study object containing optimization results.
    """    
    
    input_dim = X_train.shape[1]
    pruner = optuna.pruners.MedianPruner() if enable_pruning else optuna.pruners.NopPruner()
    study = optuna.create_study(
        direction="maximize",
        study_name="ResNet_Optimization",
        pruner=pruner,
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(
        lambda trial: optuna_objective(
            trial,
            X_train, y_train,
            X_val, y_val,
            smote_type,
            device,
            input_dim,
            scoring_metric=scoring_metric,
            enable_pruning=enable_pruning
        ),
        n_trials=n_trials,
        show_progress_bar=True
    )
    return study


In [175]:
def train_best_resnet(study, X_train, y_train, X_val=None, y_val=None,
                      smote_type="SMOTE", device="CPU",scoring_metric=None):
    """
    Train a ResNet-based classification pipeline using the best
    hyperparameters found by an Optuna study.

    The model is trained on the combined training and validation
    data if a validation set is provided.

    Args:
        study: Optuna study object containing the best hyperparameters.
        X_train: Training feature matrix.
        y_train: Training target labels.
        X_val (optional): Validation feature matrix.
        y_val (optional): Validation target labels.
        smote_type: SMOTE strategy used in the pipeline.
        device: Computation device ('CPU' or 'GPU').
        scoring_metric: Evaluation metric (currently unused).

    Returns:
        pipeline: Trained scikit-learn compatible pipeline.
    """
    best_params = study.best_params
    input_dim = X_train.shape[1]

    pipeline = build_pipeline(
        smote_type=smote_type,
        categorical_features=[],
        device=device,
        input_dim=input_dim
    )

    pipeline.set_params(**{f"resnet__{k}": v for k, v in best_params.items()})

    if X_val is not None and y_val is not None:
        X_combined = np.vstack([X_train, X_val])
        y_combined = np.hstack([y_train, y_val])
    else:
        X_combined = X_train
        y_combined = y_train

    pipeline.fit(X_combined, y_combined)

    return pipeline

In [176]:
def evaluate_and_plot(best_pipeline, X_train_sel, X_test_sel, y_test, selected_feat_names, out_path, plots):
    """
    Evaluate a trained pipeline on the test set, save metrics and model,
    and generate specified plots.

    Args:
        best_pipeline: Trained scikit-learn compatible pipeline.
        X_train_sel: Selected training features (currently unused).
        X_test_sel: Selected test features.
        y_test: True labels for the test set.
        selected_feat_names: List of selected feature names.
        out_path: Folder path to save metrics, model, and plots.
        plots: List of plots to generate. Options include
            'roc', 'confusion', 'pr'.

    Returns:
        metrics: Dictionary containing evaluation metrics on the test set.
    """
    metrics = evaluate_model(best_pipeline, X_test_sel, y_test)
    save_model(out_path, best_pipeline)
    with open(os.path.join(out_path, 'metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=2)

    if 'roc' in plots:
        plot_roc(y_test, best_pipeline.predict_proba(X_test_sel)[:,1], out_path)
    if 'confusion' in plots:
        plot_confusion(y_test, best_pipeline.predict(X_test_sel), out_path)
    if 'pr' in plots:
        plot_pr_curve(y_test, best_pipeline.predict_proba(X_test_sel)[:,1], out_path)

    return metrics

In [177]:
def calculate_cv_metrics(model, X, y, cv=5):
    """
    Calculate multiple classification performance metrics using stratified
    cross-validation and return the mean and standard deviation for each metric.

    The function evaluates the model on each fold and computes discrimination,
    calibration-related, and diagnostic metrics based on predicted labels and
    probabilities.

    Args:
        model: Scikit-learn compatible classifier implementing fit, predict,
            and predict_proba.
        X: Feature matrix.
        y: Target labels.
        cv: Number of stratified cross-validation folds.

    Returns:
        results: Dictionary where each key is a metric name and each value
        is a dictionary containing the mean and standard deviation across folds.
    """

    if hasattr(X, "values"):
        X = X.values
    if hasattr(y, "values"):
        y = y.values

    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    metrics = {
        "roc_auc": [],
        "accuracy": [],
        "sensitivity_recall": [],
        "specificity": [],
        "precision": [],
        "balanced_accuracy": [],
        "f2_score": [],
        "youden_j_statistic": [],
        "lr_positive": [],
        "lr_negative": [],
        "diagnostic_odds_ratio": []
    }

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        sensitivity = recall_score(y_test, y_pred, pos_label=1)
        specificity = recall_score(y_test, y_pred, pos_label=0)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        j_stat = sensitivity + specificity - 1
        lr_pos = sensitivity / (1 - specificity) if specificity < 1 else np.inf
        lr_neg = (1 - sensitivity) / specificity if specificity > 0 else np.inf
        dor = lr_pos / lr_neg if lr_neg != 0 else np.inf

        metrics["roc_auc"].append(roc_auc_score(y_test, y_prob))
        metrics["accuracy"].append(accuracy_score(y_test, y_pred))
        metrics["precision"].append(precision_score(y_test, y_pred))
        metrics["sensitivity_recall"].append(sensitivity)
        metrics["specificity"].append(specificity)
        metrics["balanced_accuracy"].append(balanced_accuracy_score(y_test, y_pred))
        metrics["f2_score"].append(fbeta_score(y_test, y_pred, beta=2))
        metrics["youden_j_statistic"].append(j_stat)
        metrics["lr_positive"].append(lr_pos)
        metrics["lr_negative"].append(lr_neg)
        metrics["diagnostic_odds_ratio"].append(dor)

    results = {
        k: {
            "mean": np.mean(v),
            "std": np.std(v)
        } for k, v in metrics.items()
    }

    return results

In [178]:
def calculate_extended_metrics(model, X, y):
    """
    Compute a comprehensive set of classification metrics for a given model.

    Args:
        model: Trained sklearn-compatible classifier with predict and predict_proba methods.
        X: Features for evaluation.
        y: True binary labels.

    Returns:
        dict: Dictionary containing various performance metrics, including:
            - auc: ROC AUC score
            - accuracy
            - sensitivity (recall for positive class)
            - specificity (recall for negative class)
            - tp, tn, fp, fn: confusion matrix counts
            - combined: average of sensitivity and specificity
            - prc_auc: Area under precision-recall curve
            - f2: F2-score
            - lr+, lr-: positive and negative likelihood ratios
            - j_stat: Youden's J statistic
            - diagnostic_odds_ratio: ratio of lr+ to lr-
    """
    metrics = {}

    y_prob = model.predict_proba(X)[:, 1]
    metrics['auc'] = roc_auc_score(y, y_prob)

    y_pred = model.predict(X)
    metrics['accuracy'] = np.mean(y_pred == y)
    metrics['sensitivity'] = recall_score(y, y_pred, pos_label=1)
    metrics['specificity'] = recall_score(y, y_pred, pos_label=0)
    
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    metrics['tp'] = tp
    metrics['tn'] = tn
    metrics['fp'] = fp
    metrics['fn'] = fn

    metrics['combined'] = (metrics['sensitivity'] + metrics['specificity']) / 2

    precision, recall, _ = precision_recall_curve(y, y_prob)
    metrics['prc_auc'] = auc(recall, precision)

    metrics['f2'] = fbeta_score(y, y_pred, beta=2)

    sens = metrics['sensitivity']
    spec = metrics['specificity']
    metrics['lr+'] = sens / (1 - spec) if spec < 1 else np.inf
    metrics['lr-'] = (1 - sens) / spec if spec > 0 else np.inf

    metrics['j_stat'] = sens + spec - 1

    metrics['diagnostic_odds_ratio'] = (
        metrics['lr+'] / metrics['lr-'] if metrics['lr-'] != 0 else np.inf
    )

    return metrics

In [179]:
def save_predictions_with_confidence(model, X_test_sel, y_test, out_path, threshold=0.5):
    """
    Generate predictions with confidence scores from a resnet model and save to CSV.

    Args:
        model: Trained resnet classifier with predict_proba method.
        X_test_sel: Test features for prediction.
        y_test: True labels corresponding to X_test_sel.
        out_path: Directory to save the CSV file.
        threshold: Probability cutoff for binary classification. Defaults to 0.5.

    Saves:
        CSV file 'predictions_with_confidence.csv' containing columns:
            - prediction: predicted class labels based on threshold
            - confidence: predicted probability for positive class
            - true_label: actual labels
    """
    y_proba = model.predict_proba(X_test_sel)[:, 1]  
    y_pred = (y_proba >= threshold).astype(int)
    
    df_out = pd.DataFrame({
        'prediction': y_pred,
        'confidence': y_proba,
        'true_label': y_test
    })
    df_out.to_csv(os.path.join(out_path, 'predictions_with_confidence.csv'), index=False)


In [180]:
def generate_threshold_confusion_matrices(out_path, custom_thresholds=None):
    """
    Generate confusion matrices and compute metrics for multiple thresholds 
    using predictions with confidence scores saved in CSV. Saves plots and 
    summaries to disk.

    Args:
        out_path: Directory containing 'predictions_with_confidence.csv' 
            and where results will be saved.
        custom_thresholds (list, optional): List of additional thresholds to 
            evaluate. Defaults to [0.1,0.2,0.3,0.4,0.6,0.7,0.8].

    Returns:
        dict: Summary of key thresholds and their Youden's J statistic, including:
            - best_threshold: threshold maximizing Youden's J
            - best_j: value of Youden's J at best_threshold
            - guaranteed_pos_threshold: minimum score for all positive cases
            - guaranteed_neg_threshold: maximum score for all negative cases
            - custom_thresholds: list of custom thresholds evaluated

    Saves:
        - Confusion matrix plots for best, guaranteed, and custom thresholds
        - CSV summary with metrics for all thresholds ('threshold_metrics_summary.csv')
        - JSON summary ('threshold_summary.json')
    """
    if custom_thresholds is None:
        custom_thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

    csv_path = os.path.join(out_path, "predictions_with_confidence.csv")
    df = pd.read_csv(csv_path)

    y_true = df["true_label"].values
    y_score = df["confidence"].values

    thresholds = np.linspace(0, 1, 1000)
    best_j = -np.inf
    best_t = 0.0
    for t in thresholds:
        y_pred = (y_score >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        sens = tp / (tp + fn) if (tp + fn) else 0
        spec = tn / (tn + fp) if (tn + fp) else 0
        j = sens + spec - 1
        if j > best_j:
            best_j = j
            best_t = t

    pos_mask = y_true == 1
    if pos_mask.sum() > 0:
        guaranteed_pos_threshold = float(np.min(y_score[pos_mask]))
    else:
        guaranteed_pos_threshold = 1.0

    neg_mask = y_true == 0
    if neg_mask.sum() > 0:
        max_neg = float(np.max(y_score[neg_mask]))
        eps = np.finfo(float).eps * 10
        guaranteed_neg_threshold = min(max_neg + eps, 1.0)
    else:
        guaranteed_neg_threshold = 0.0

    threshold_dir = os.path.join(out_path, "thresholded")
    os.makedirs(threshold_dir, exist_ok=True)

    def plot_cm(y_pred, thr, name, filename):
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(4,3))
        plt.imshow(cm, cmap="Blues")
        plt.title(f"{name}\nThreshold={thr:.6f}")
        plt.colorbar()
        plt.xticks([0,1], ["neg","pos"])
        plt.yticks([0,1], ["neg","pos"])
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, int(cm[i,j]), ha="center", va="center")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(threshold_dir, filename), dpi=300)
        plt.close()

    def compute_metrics(y_true, y_pred, y_prob=None):

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        sens = tp / (tp + fn) if (tp + fn) else 0
        spec = tn / (tn + fp) if (tn + fp) else 0
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        youden_j = sens + spec - 1

        lr_pos = sens / (1 - spec) if spec < 1 else np.inf
        lr_neg = (1 - sens) / spec if spec > 0 else np.inf
        dor = lr_pos / lr_neg if lr_neg != 0 else np.inf

        roc_auc = roc_auc_score(y_true, y_prob) if y_prob is not None else None

        return {
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "accuracy": acc,
            "sensitivity_recall": sens,
            "specificity": spec,
            "precision": prec,
            "F1": f1,
            "f2_score": f2,
            "balanced_accuracy": bal_acc,
            "youden_j_statistic": youden_j,
            "lr_positive": lr_pos,
            "lr_negative": lr_neg,
            "diagnostic_odds_ratio": dor,
            "roc_auc": roc_auc
        }

    thresholds_info = {
        "best_threshold": best_t,
        "guaranteed_positives": guaranteed_pos_threshold,
        "guaranteed_negatives": guaranteed_neg_threshold
    }

    summary_rows = []
    for name, t in thresholds_info.items():
        y_pred = (y_score >= t).astype(int)
        plot_cm(y_pred, t, name.replace("_", " ").title(), f"cm_{name}.png")
        metrics = compute_metrics(y_true, y_pred, y_prob=y_score)
        metrics["threshold"] = t
        metrics["type"] = name
        summary_rows.append(metrics)

    for t in custom_thresholds:
        y_pred = (y_score >= t).astype(int)
        plot_cm(y_pred, t, f"Custom Threshold {t}", f"cm_custom_{t:.2f}.png")
        metrics = compute_metrics(y_true, y_pred, y_prob=y_score)
        metrics["threshold"] = t
        metrics["type"] = "custom"
        summary_rows.append(metrics)

    df_summary = pd.DataFrame(summary_rows)
    csv_summary_path = os.path.join(threshold_dir, "threshold_metrics_summary.csv")
    df_summary.to_csv(csv_summary_path, index=False)

    summary_json = {
        "best_threshold": float(best_t),
        "best_j": float(best_j),
        "guaranteed_pos_threshold": float(guaranteed_pos_threshold),
        "guaranteed_neg_threshold": float(guaranteed_neg_threshold),
        "custom_thresholds": custom_thresholds
    }
    with open(os.path.join(threshold_dir, "threshold_summary.json"), "w") as f:
        json.dump(summary_json, f, indent=2)

    return summary_json


In [181]:
def bootstrap_model_performance(model, X_test, y_test, threshold=0.5, n_bootstrap=2000, n_jobs=-1, random_state=42):
    """
    Estimate model performance metrics with bootstrapped confidence intervals using parallel processing.

    Args:
        model: Trained classifier with predict_proba and predict methods.
        X_test: Test feature set.
        y_test: True binary labels for X_test.
        threshold: Probability cutoff for binary predictions.
        n_bootstrap: Number of bootstrap resamples.
        n_jobs: Number of parallel jobs. -1 uses all cores.
        random_state: Random seed.

    Returns:
        Bootstrapped metrics (mean, std, 95% CI) like before.
    """
    rng = np.random.RandomState(random_state)
    y_proba = model.predict_proba(X_test)[:, 1]
    N = len(y_test)

    def single_bootstrap(seed):
        local_rng = np.random.RandomState(seed)
        idx = local_rng.randint(0, N, N)
        y_bs = y_test[idx]
        proba_bs = y_proba[idx]
        pred_bs = (proba_bs >= threshold).astype(int)

        TP = np.sum((pred_bs == 1) & (y_bs == 1))
        TN = np.sum((pred_bs == 0) & (y_bs == 0))
        FP = np.sum((pred_bs == 1) & (y_bs == 0))
        FN = np.sum((pred_bs == 0) & (y_bs == 1))

        sens = TP / (TP + FN) if (TP + FN) > 0 else np.nan
        spec = TN / (TN + FP) if (TN + FP) > 0 else np.nan
        lr_pos = sens / (1 - spec) if (1 - spec) > 0 else np.nan
        lr_neg = (1 - sens) / spec if spec > 0 else np.nan
        dor = lr_pos / lr_neg if (lr_pos > 0 and lr_neg > 0) else np.nan

        return {
            "roc_auc": roc_auc_score(y_bs, proba_bs),
            "accuracy": accuracy_score(y_bs, pred_bs),
            "sensitivity_recall": sens,
            "specificity": spec,
            "precision": precision_score(y_bs, pred_bs, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_bs, pred_bs),
            "f2_score": fbeta_score(y_bs, pred_bs, beta=2, zero_division=0),
            "youden_j_statistic": sens + spec - 1,
            "lr_positive": lr_pos,
            "lr_negative": lr_neg,
            "diagnostic_odds_ratio": dor
        }

    results = Parallel(n_jobs=n_jobs)(
        delayed(single_bootstrap)(rng.randint(0, 1_000_000)) for _ in range(n_bootstrap)
    )

    def stats(metric):
        arr = np.array([r[metric] for r in results], dtype=float)
        return {
            "mean": float(np.nanmean(arr)),
            "std": float(np.nanstd(arr)),
            "ci_95": (
                float(np.nanpercentile(arr, 2.5)),
                float(np.nanpercentile(arr, 97.5))
            )
        }

    metrics_list = [
        "roc_auc", "accuracy", "sensitivity_recall", "specificity", "precision",
        "balanced_accuracy", "f2_score", "youden_j_statistic",
        "lr_positive", "lr_negative", "diagnostic_odds_ratio"
    ]

    return {metric: stats(metric) for metric in metrics_list} | {"threshold": threshold, "n_bootstrap": n_bootstrap}

In [182]:
def train_best_resnet_calibrated(study, X_train, y_train, X_val=None, y_val=None,
                                 smote_type="SMOTE", device="CPU", scoring_metric=None):
    """
    Train a ResNet-based classification pipeline using the best hyperparameters
    from an Optuna study and optionally calibrate it on a validation set.

    If a validation set is provided, the ResNet model is calibrated using
    sigmoid calibration (Platt scaling) with CalibratedClassifierCV.

    Args:
        study: Optuna study object containing the best hyperparameters.
        X_train: Training feature matrix.
        y_train: Training target labels.
        X_val (optional): Validation feature matrix for calibration.
        y_val (optional): Validation target labels for calibration.
        smote_type: SMOTE strategy used in the pipeline.
        device: Computation device ('CPU' or 'GPU').
        scoring_metric: Evaluation metric (currently unused).

    Returns:
        pipeline: Trained and optionally calibrated scikit-learn compatible pipeline.
    """
    best_params = study.best_params
    input_dim = X_train.shape[1]

    pipeline = build_pipeline(
        smote_type=smote_type,
        categorical_features=[],
        device=device,
        input_dim=input_dim
    )

    pipeline.set_params(**{f"resnet__{k}": v for k, v in best_params.items()})

    if X_val is not None and y_val is not None:
        X_combined = np.vstack([X_train, X_val])
        y_combined = np.hstack([y_train, y_val])
    else:
        X_combined = X_train
        y_combined = y_train

    pipeline.fit(X_combined, y_combined)

    if X_val is not None and y_val is not None:
        calibrated_resnet = CalibratedClassifierCV(
            pipeline.named_steps['resnet'], 
            method='sigmoid', 
            cv='prefit'
        )
        calibrated_resnet.fit(X_val, y_val)

        pipeline.named_steps['resnet'] = calibrated_resnet

    return pipeline

In [183]:
def run_experiments_resnet(
    df, feature_columns, target_col,
    outdir='resnet_runs',
    split_mode='validation',
    n_trials=200
):
    """
    Run full ResNet experiments including data splitting, Optuna hyperparameter
    optimization, model training, calibration, cross-validation, bootstrap
    evaluation, and result saving.

    The function loops over SMOTE types and class weights, trains a calibrated
    ResNet for each configuration, computes metrics, generates plots, and
    saves all outputs to the specified directory.

    Args:
        df: Pandas DataFrame containing features and target column.
        feature_columns: List of column names to use as input features.
        target_col: Name of the target column in the DataFrame.
        outdir: Directory to save results for all experiments.
        split_mode: Mode for splitting data; 'validation' uses a separate
            validation set, otherwise only train/test split is applied.
        weights: List of positive class weights to experiment with.
        n_trials: Number of Optuna trials for hyperparameter optimization.

    Returns:
        None. All results are saved to 'outdir', including:
            - Best hyperparameters per configuration
            - Cross-validation metrics
            - Bootstrap performance metrics
            - ROC, PR, and confusion plots
            - Extended metrics and prediction files
            - Summary CSV of all experiments
    """
    os.makedirs(outdir, exist_ok=True)
    summary_rows = []

    X = df[feature_columns].values
    y = df[target_col].values

    if split_mode == 'validation':
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=0.125, stratify=y_temp, random_state=42
        )
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )
        X_val, y_val = None, None

    for smote_type in smote_types:

        folder_name = f"resnet_{smote_type.lower()}_{str().replace('.', '_')}"
        out_path = os.path.join(outdir, folder_name)
        os.makedirs(out_path, exist_ok=True)

        X_train_sel, X_val_sel = X_train, X_val
        selected_feat_names = feature_columns

        study = run_optuna_resnet(
            X_train_sel, y_train,
            X_val_sel, y_val,
            smote_type=smote_type,
            device=device_choice,
            n_trials=n_trials,
            scoring_metric=scoring_metric
        )

        best_params_path = os.path.join(out_path, "best_hyperparameters.json")
        best_info = {
            "best_params": study.best_params,
            "best_value": study.best_value
        }
        with open(best_params_path, "w") as f:
            json.dump(best_info, f, indent=2)
        print(f"Best hyperparameters saved to: {best_params_path}")

        best_pipeline = train_best_resnet_calibrated(
            study,
            X_train_sel, y_train,
            X_val_sel, y_val,
            smote_type=smote_type,
            device=device_choice,
            scoring_metric=scoring_metric,
        )


        calibrated_pipeline = best_pipeline
        X_combined = np.vstack([X_train_sel, X_val_sel]) if X_val_sel is not None else X_train_sel
        y_combined = np.hstack([y_train, y_val]) if y_val is not None else y_train

        cv_metrics = calculate_cv_metrics(calibrated_pipeline, X_train_sel, y_train)
        with open(os.path.join(out_path, "cv_metrics.json"), "w") as f:
            json.dump(cv_metrics, f, indent=2)

        X_test_sel = X_test[:, [feature_columns.index(f) for f in selected_feat_names]]

        bootstrap_results = bootstrap_model_performance(
            calibrated_pipeline,
            X_test_sel,
            y_test,
            n_bootstrap=2000,
            n_jobs=-1
        )
        with open(os.path.join(out_path, "bootstrap_metrics.json"), "w") as f:
            json.dump(bootstrap_results, f, indent=2)
        print("Bootstrap CI opgeslagen.")

        evaluate_and_plot(calibrated_pipeline, X_combined, X_test_sel, y_test, selected_feat_names, out_path, plots)
        metrics = calculate_extended_metrics(calibrated_pipeline, X_test_sel, y_test)
        save_predictions_with_confidence(calibrated_pipeline, X_test_sel, y_test, out_path, threshold=0.5)
        generate_threshold_confusion_matrices(out_path)

        summary_rows.append({
            **metrics,
            'folder': folder_name,
            'smote_type': smote_type,
            'n_features': len(selected_feat_names),
            'split_mode': split_mode
        })


    df_summary = pd.DataFrame(summary_rows).sort_values(by=['auc', 'combined'], ascending=False)
    df_summary.to_csv(os.path.join(outdir, 'summary.csv'), index=False)
    print(f"All ResNet models have been trained and results saved in '{outdir}'")


In [184]:
def safe_input(prompt, valid_values=None, default=None, type_func=str, allow_multiple=False):
    """
    Prompt the user for input safely with optional validation, default values, and type conversion.

    Args:
        prompt: The message to display to the user.
        valid_values (optional): List of allowed values. If provided, input not in this list
            will fall back to the default.
        default: The default value to return if the user enters nothing or invalid input.
        type_func: Function to convert the input string to a desired type
            (e.g., int, float, str). Defaults to str.
        allow_multiple: If True, allows multiple values separated by spaces, 
            returning a list of converted items. Defaults to False.

    Returns:
        single value or list: The user input converted to the desired type, or default if invalid/missing.
    
    Notes:
        - If allow_multiple=True, the input string is split on spaces, each item is type-converted, 
          and checked against valid_values if provided.
        - If the user input is empty, the default is returned.
        - If conversion or validation fails, the default is returned and a message is printed.
    """
    val = input(f"{prompt} [default: {default}]: ").strip()
    if not val:
        return default
    try:
        if allow_multiple:
            items = [type_func(v) for v in val.split()]
            if valid_values:
                items = [v if v in valid_values else default for v in items]
            return items
        else:
            typed_val = type_func(val)
            if valid_values and typed_val not in valid_values:
                print(f"Invalid input '{typed_val}', using default '{default}'.")
                return default
            return typed_val
    except Exception as e:
        print(f"Invalid input '{val}' ({e}), using default '{default}'.")
        return default
    
    
device_choice = safe_input(
    "Which device do you want to use? (CPU / GPU)",
    valid_values=['CPU', 'GPU'],
    default='CPU',
    type_func=str
)

split_mode = safe_input(
    "Do you want to use a validation set? (none / validation)",
    valid_values=['none', 'validation'],
    default='validation'
)

fs_input = safe_input(
    "Which feature selection method? (NONE)",
    default='NONE',
    type_func=str,
    allow_multiple=True
)
feature_methods = [s.upper() for s in fs_input] if isinstance(fs_input, list) else fs_input.upper().split()
if 'ALL' in feature_methods:
    feature_methods = ['NONE']


plots_input = safe_input(
    "Which results/plots do you want? (roc, pr, confusion, all)",
    default='roc pr confusion',
    type_func=str,
    allow_multiple=True
)
plots = [p.lower() for p in plots_input] if isinstance(plots_input, list) else plots_input.lower().split()
if 'all' in plots:
    plots = ['roc', 'pr', 'confusion']



scoring_metric = safe_input(
    "Which scoring metric do you want to use? "
    "(roc_auc / average_precision / f2 / balanced_accuracy / mcc / f1 / accuracy / logloss / recall)",
    valid_values=['roc_auc', 'average_precision', 'f2', 'balanced_accuracy',
                  'mcc', 'f1', 'accuracy', 'logloss','recall'],
    default='roc_auc',
    type_func=str
)

n_trials = safe_input(
    "Hoeveel Optuna trials wil je uitvoeren?",
    default=200,
    type_func=int
)

smote_types = safe_input(
    "Which SMOTE version do you want to use? (NONE SMOTE BorderlineSMOTE ADASYN SMOTENC)",
    default=['NONE'],
    type_func=str,
    allow_multiple=True
)

smote_types = [s.upper() for s in smote_types]
smote_types = ['NONE' if s == "NONE" else s for s in smote_types]


In [ ]:
df = pd.read_csv('final_training.csv')
target_col = 'status' 
feature_columns = [c for c in df.columns if c != target_col]

In [ ]:
run_experiments_resnet(df, feature_columns, target_col, outdir='resnet_sysmex_runs', split_mode=split_mode, n_trials=n_trials)
print("All models are trained and saved in 'resnet_sysmex_runs'.")